# Visual History Agent — 01 Formal Evidence Agent

Requires a trained global dictionary (`00_dictionary_learning.ipynb`).

1. **Qwen RCS cache** — score catalogue description chunks for painterly evidence  
2. **Cloud atom synthesis** — rerank and label each dictionary atom

tqdm covers retrieval assembly, RCS chunk scoring, and atom synthesis.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path("../").resolve()
os.environ["VISUAL_HISTORY_AGENT_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from visual_history_agent.paths import artifact_paths, require_file

paths = artifact_paths(ROOT)
require_file(paths["dictionary"], label="dictionary atoms")
require_file(paths["embedding_mean"], label="embedding mean")
require_file(paths["database"], label="painting database")
print("Project:", ROOT)
print("Dictionary:", paths["dictionary"])
print("Atom output dir:", paths["atom_descriptions_dir"])


## 1 — Qwen Retrieval-augmented Contextual Summarization (RCS) cache

What RCS (Local Ollama `qwen2.5:7b` by default) is doing

For each of the 1000 dictionary atoms:

1) Take the atom’s top-40 nearest paintings (cosine in mean-centered 2304-D).
2) Split each painting’s catalogue description into overlapping text windows (~180 tokens, 60 overlap) via VisualHistory’s chunker.
3) For every unique chunk, call local Ollama qwen2.5:7b and ask:
4) “How useful is this passage as concrete painterly evidence?” → JSON with relevance_score 1–10 + 1–8 evidence sentences.
5) Append results to qwen_rcs_chunk_cache.jsonl (content-hash keyed, so repeats across atoms are scored once).


In [ ]:
from run_formal_evidence_agent import main as run_formal_evidence_agent

out_dir = str(paths["atom_descriptions_dir"])
run_formal_evidence_agent([
    "--rcs-only",
    "--output-dir", out_dir,
])
print("RCS cache:", paths["qwen_cache"])


## 2 — Atom synthesis (OpenAI GPT-5.1)

Requires complete RCS coverage from the cell above. Uses direct OpenAI (`gpt-5.1`), not Azure.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path("../").resolve()
os.environ["VISUAL_HISTORY_AGENT_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from visual_history_agent.paths import artifact_paths
from run_formal_evidence_agent import main as run_formal_evidence_agent

paths = artifact_paths(ROOT)
out_dir = str(paths["atom_descriptions_dir"])

run_formal_evidence_agent([
    "--synthesis-only",
    "--output-dir", out_dir,
    "--coordinator-backend", "openai",
    "--coordinator-model", "gpt-5.1",
])
print("Atom labels:", paths["atom_descriptions"])
